# TP 1 — Premier notebook géospatial

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP1_premier_notebook_geospatial.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 1 · KOUADIO Kouao Laurent

Objectifs :
1. Utiliser un environnement Python (Google Colab ou conda).
2. Charger des données vecteur ouvertes sur la Côte d'Ivoire.
3. Vérifier et changer le système de coordonnées (SCR).
4. Calculer des superficies correctes et produire une carte.
5. Estimer un ordre de grandeur de Big Data (Sentinel-2).

Durée : environ 75 minutes.

## 1. Installation

Sur Google Colab, exécutez la cellule ci-dessous. Dans l'environnement
conda du cours, les paquets sont déjà installés.

In [ ]:
%pip install -q geopandas folium mapclassify

In [ ]:
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

print("GeoPandas", gpd.__version__)

## 2. Charger les limites administratives

La base [geoBoundaries](https://www.geoboundaries.org) fournit des
limites administratives ouvertes. En Côte d'Ivoire, le niveau **ADM1**
correspond aux **14 districts** (dont les districts autonomes d'Abidjan
et de Yamoussoukro) ; les 31 régions sont au niveau ADM2.

In [ ]:
url = ("https://www.geoboundaries.org/api/"
       "current/gbOpen/CIV/ADM1/")
meta = requests.get(url, timeout=60).json()
civ = gpd.read_file(meta["gjDownloadURL"])

civ.head()

**Question 1.** Combien de lignes contient `civ` ? Que représente la
colonne `geometry` ?

In [ ]:
print("Nombre de districts :", len(civ))
print("SCR :", civ.crs)
print("Emprise (lon_min, lat_min, lon_max, lat_max) :")
print(civ.total_bounds)

## 3. Une première carte statique

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
civ.plot(ax=ax, color="#97BC62", edgecolor="white", linewidth=0.8)
civ.boundary.plot(ax=ax, color="#1B4332", linewidth=0.5)
ax.set_title("Districts de Côte d'Ivoire (geoBoundaries ADM1)")
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
plt.show()

## 4. Pourquoi le SCR compte : calcul de superficie

Les coordonnées sont en degrés (EPSG:4326). Une « surface » calculée
directement en degrés carrés n'a pas de sens physique : GeoPandas
affiche d'ailleurs un avertissement.

In [ ]:
civ.area.head()  # degrés carrés : à ne pas utiliser

On reprojette dans une **projection à aires égales** adaptée à
l'Afrique : *Africa Albers Equal Area Conic* (`ESRI:102022`). Les
unités deviennent des mètres.

In [ ]:
civ_aea = civ.to_crs("ESRI:102022")
civ["km2"] = civ_aea.area / 1e6

(civ[["shapeName", "km2"]]
 .sort_values("km2", ascending=False)
 .round(0))

In [ ]:
total = civ["km2"].sum()
print(f"Superficie totale : {total:,.0f} km2")

**Question 2.** La superficie officielle du pays est d'environ
322 000 km². Comparez avec votre résultat. D'où peuvent venir les
écarts ?

**Question 3.** Refaites le calcul en UTM zone 30N (`EPSG:32630`).
Quel district change le plus ? Pourquoi (indice : le méridien 6° Ouest
sépare les zones UTM 29 et 30) ?

In [ ]:
civ["km2_utm30"] = civ.to_crs("EPSG:32630").area / 1e6
civ["ecart_%"] = 100 * (civ["km2_utm30"] - civ["km2"]) / civ["km2"]
civ[["shapeName", "km2", "km2_utm30", "ecart_%"]].round(2)

## 5. Carte interactive

In [ ]:
civ.explore(column="km2", cmap="YlGn", tooltip=["shapeName", "km2"],
            legend_kwds={"caption": "Superficie (km2)"})

## 6. Exercice : ordre de grandeur du Big Data Sentinel-2

Reprenez le calcul du cours avec des variables. Une tuile Sentinel-2
couvre 100 km × 100 km ; la revisite est de 5 jours.

In [ ]:
superficie_km2 = total          # calculée ci-dessus
surface_tuile_km2 = 100 * 100
facteur_recouvrement = 1.2      # hypothèse : tuiles en bord de pays
taille_tuile_go = 0.8           # hypothèse : produit L2A compressé
revisite_jours = 5

n_tuiles = superficie_km2 / surface_tuile_km2 * facteur_recouvrement
n_acq = 365 / revisite_jours
volume_to = n_tuiles * n_acq * taille_tuile_go / 1000

print(f"Tuiles : {n_tuiles:.0f}")
print(f"Acquisitions par an et par tuile : {n_acq:.0f}")
print(f"Volume annuel : {volume_to:.1f} To")

**Question 4.** Quel paramètre est le plus incertain ? Recherchez sur
le Copernicus Data Space Ecosystem la taille réelle d'un produit L2A
sur la Côte d'Ivoire et corrigez l'estimation.

**Question 5.** Combien d'années d'archives Sentinel-2 tiendraient sur
le disque de votre ordinateur ? Qu'en concluez-vous pour la suite du
cours ?

## À rendre

Le notebook exécuté, avec vos réponses aux questions 1 à 5 dans des
cellules Markdown.